# Musanga V2 — Notebook 02 | Frozen U-Net crown evaluation and illumination sensitivity

**Read-only CPU benchmark.** Tests the frozen production U-Net against the independent merged Luki crown labels (179 Musanga, 185 non-Musanga), and characterizes probability/illumination relationships at all five UAV sites. Uses existing **native-resolution UAV probability maps** and raster geometry, with no retraining or new full-mosaic inference.

Run after V2 Notebooks 00 and 01. Every output goes to `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/02_frozen_baseline/`.

**Scientific safeguards:** The production U-Net was trained on MOS, Yoko, Yangambi and Baego. Any metrics computed there are **apparent training-domain diagnostics**, not independent site transfer results. **Luki is the genuine independent evaluation**. Use both training-pooled and *site-relative* illumination strata because Luki is globally brighter than the other forests. Crown-level segmentation labels are used only inside annotated crowns, not as whole-scene reference masks.

Sections are numbered and independently inspectable after the setup cell.

## 00 — Source contract and evaluation safeguards

CPU-only. This notebook reuses the **frozen full-resolution probability maps**; it does not rerun the U-Net, alter maps or retrain. Luki is an independent external evaluation site. The original four training forests were included in the frozen production U-Net, so their metrics are **apparent/in-sample** and cannot be presented as independent validation.

In [ ]:
%pip -q install geopandas pyogrio rasterio shapely scipy scikit-learn matplotlib pandas
from pathlib import Path
import json,math,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio,matplotlib.pyplot as plt
from rasterio.features import geometry_mask
from rasterio.windows import Window,from_bounds
from scipy.stats import spearmanr
from sklearn.metrics import (roc_auc_score,average_precision_score,balanced_accuracy_score,
    precision_score,recall_score,f1_score,confusion_matrix,brier_score_loss)
from IPython.display import display
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
OUT=V2/"02_frozen_baseline";OUT.mkdir(parents=True,exist_ok=True)
REF=V2/"01_reference_splits";MERGED=REF/"01_Luki_frozen_external_reference.gpkg"
MAPS=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"
assert MERGED.exists(),f"Run V2 Notebook 01, section 02: {MERGED}"
LUKI_VERSION_MANIFEST=REF/"01_luki_current_reference_version.json"
assert LUKI_VERSION_MANIFEST.exists(),"Run updated V2 Notebook 01 Section 02 first"
LUKI_REFERENCE=json.loads(LUKI_VERSION_MANIFEST.read_text())
LUKI_VERSION=LUKI_REFERENCE["reference_id"]
# Keep old baseline outputs untouched; new reference versions get separate folders.
OUT=OUT/"by_luki_reference"/LUKI_VERSION
OUT.mkdir(parents=True,exist_ok=True)
print("Using immutable external Luki reference version",LUKI_VERSION)

assert (REF/"01_crown_illumination.csv").exists(),f"Run V2 Notebook 01, section 05."
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp"),
 "Luki":("Luki.tif",None)}
MIN_AREA=5.0;P_SCALE=10000.;P_NODATA=65535
CROWN_SCORE_THRESHOLD=.50  # classification from mean U-Net probability WITHIN annotated crown
PIXEL_COVER_THRESHOLD=.50  # descriptive fraction of predicted-Musanga pixels within each crown
MAX_CROWN_WINDOW=2048
MIN_CROWN_VALID_PIXELS=10
print("V2 frozen diagnostic output:",OUT)
print("Scope: external Luki evaluation and descriptive TRAINING-site diagnostics, no fitting.")


## 01 — Reconstruct the immutable crown reference

Read precisely the same eligible polygons used in Notebook 01. Cross-check IDs against saved split registers and use the merged **364-crown Luki** reference. No label source is silently added or omitted.

In [ ]:
CROWNS={}
for site,(rgb,ann) in SITES.items():
    if site=="Luki":
        g=gpd.read_file(MERGED,layer="external_evaluation")
        assert len(g)==int(LUKI_REFERENCE["crowns"]), "Luki reference metadata disagrees with dataset"
        assert int((g.class_id==1).sum())==int(LUKI_REFERENCE["musanga"])
        g["label"]=(g.class_id==1).astype(int)
        g["crown_id"]=g["crown_id"].astype(str)
        g["split"]="external"
    else:
        g=gpd.read_file(AD/ann)
        with rasterio.open(GEO/rgb) as ds:
            assert g.crs is not None
            if g.crs!=ds.crs:g=g.to_crs(ds.crs)
            footprint=__import__("shapely.geometry",fromlist=["box"]).box(*ds.bounds)
        valid=(g.geometry.notna())&(~g.geometry.is_empty)&g.geometry.is_valid
        g=g.loc[valid].copy()
        class_ids=pd.to_numeric(g.Class_ID,errors="coerce")
        g=g.loc[class_ids.notna()].copy()
        g["class_id"]=class_ids.loc[g.index].astype(int)
        g["area_m2"]=g.geometry.area
        g=g.loc[(g.area_m2>=MIN_AREA)&g.geometry.intersects(footprint)].copy()
        g["source_row"]=g.index.astype(str)
        g["crown_id"]=site+":"+g.source_row
        g["label"]=(g.class_id==1).astype(int)
        register=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
        assert set(g.crown_id)==set(register.crown_id),f"{site}: source annotation/split register changed"
        g=g.merge(register[["crown_id","split"]],on="crown_id",how="left",validate="one_to_one")
        assert g.split.notna().all()
    assert g.crown_id.is_unique
    CROWNS[site]=g
    print(site,"reference crowns:",len(g),"Musanga:",int(g.label.sum()),flush=True)
illum=pd.read_csv(REF/"01_crown_illumination.csv",dtype={"crown_id":"string"})
assert illum.crown_id.is_unique, "Duplicate illumination rows in Notebook 01 outputs"


## 02 — Crown-aligned frozen U-Net probability audit

Read one bounded UAV window per crown from the established full-resolution probability map. Report **mean predicted probability**, **fraction of crown pixels above 0.5**, number of valid prediction pixels and reference class. Class evaluation uses the mean probability; fractional cover is a separate diagnostic.

In [ ]:
def crown_stats02(geom,src):
    window=from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
    x0=max(0,int(window.col_off));y0=max(0,int(window.row_off))
    x1=min(src.width,int(math.ceil(window.col_off+window.width)))
    y1=min(src.height,int(math.ceil(window.row_off+window.height)))
    w=x1-x0;h=y1-y0
    if w<=0 or h<=0:return dict(mean_p=np.nan,cover_p05=np.nan,valid_px=0,coverage=np.nan,qa="outside")
    if w>MAX_CROWN_WINDOW or h>MAX_CROWN_WINDOW:
        return dict(mean_p=np.nan,cover_p05=np.nan,valid_px=0,coverage=np.nan,qa="window_too_large")
    win=Window(x0,y0,w,h)
    within=geometry_mask([geom],out_shape=(h,w),transform=src.window_transform(win),invert=True)
    denom=int(within.sum())
    if denom<1:return dict(mean_p=np.nan,cover_p05=np.nan,valid_px=0,coverage=0,qa="subpixel_geometry")
    z=src.read(1,window=win)
    v=within&(z!=P_NODATA)&(z>=0)&(z<=P_SCALE)
    count=int(v.sum())
    coverage=count/denom
    if count<MIN_CROWN_VALID_PIXELS or coverage<.90:
        return dict(mean_p=np.nan,cover_p05=np.nan,valid_px=count,coverage=coverage,qa="insufficient_valid")
    p=z[v].astype(np.float32)/P_SCALE
    return dict(mean_p=float(p.mean()),cover_p05=float(np.mean(p>=PIXEL_COVER_THRESHOLD)),
                valid_px=count,coverage=coverage,qa="ok")

ROWS=[]
for site,g in CROWNS.items():
    srcpath=MAPS/f"{site}_musanga_probability_uint16_FULL.tif"
    assert srcpath.exists(),f"Missing frozen probability map: {srcpath}"
    with rasterio.open(GEO/SITES[site][0]) as rgb,rasterio.open(srcpath) as probability:
        assert rgb.transform==probability.transform and rgb.crs==probability.crs
        assert rgb.width==probability.width and rgb.height==probability.height
        assert probability.count==1 and probability.dtypes[0]=="uint16"
        for _,r in g.iterrows():
            stats=crown_stats02(r.geometry,probability)
            ROWS.append(dict(site=site,crown_id=str(r.crown_id),label=int(r.label),
                source_dataset=str(r.get("source_dataset","training")),
                split=str(r.split),**stats))
    print("Frozen crown predictions read:",site,len(g),flush=True)
CROWN_PRED=pd.DataFrame(ROWS)
CROWN_PRED=CROWN_PRED.merge(illum[["site","crown_id","median_luminance",
                                 "illumination_stratum","valid_rgb_pixels"]],
                          on=["site","crown_id"],how="left",validate="one_to_one")
assert CROWN_PRED.median_luminance.notna().any()
CROWN_PRED.to_csv(OUT/"02_frozen_crown_predictions.csv",index=False)
display(pd.crosstab(CROWN_PRED.site,CROWN_PRED.qa))


## 03 — Absolute and site-relative illumination strata

The **training-pooled thresholds classify nearly all Luki crowns as bright**, making them unsuitable for diagnosing its internal cloud-shadow patches. Report both the fixed training-reference strata and **within-site brightness quartiles**, plus continuous brightness. Quartiles are defined without looking at model accuracy.

In [ ]:
CROWN_PRED["site_relative_light"]="unknown"
LIGHT_CUTS=[]
for site,g in CROWN_PRED.groupby("site"):
    valid=g.median_luminance.dropna()
    q1,q3=valid.quantile([.25,.75]).to_numpy()
    LIGHT_CUTS.append(dict(site=site,n_brightness=len(valid),q25=float(q1),
                           q75=float(q3),site_median=float(valid.median())))
    mask=(CROWN_PRED.site==site)&CROWN_PRED.median_luminance.notna()
    vals=CROWN_PRED.loc[mask,"median_luminance"]
    CROWN_PRED.loc[mask,"site_relative_light"]=np.where(vals<=q1,"darkest_25pct",
         np.where(vals>=q3,"brightest_25pct","middle_50pct"))
LIGHT_CUTS=pd.DataFrame(LIGHT_CUTS)
LIGHT_CUTS.to_csv(OUT/"02_site_specific_illumination_cutoffs.csv",index=False)
CROWN_PRED.to_csv(OUT/"02_frozen_crown_predictions.csv",index=False)
display(LIGHT_CUTS.round(3))
display(pd.crosstab(CROWN_PRED.site,CROWN_PRED.site_relative_light))
display(pd.crosstab(CROWN_PRED.site,CROWN_PRED.illumination_stratum))


## 04 — Species-discrimination metrics with external-test designation

Calculate ROC-AUC, average precision, precision/recall, balanced accuracy, F1, Brier score and confusion counts at crown level. **Only Luki is externally independent for this frozen model.** Small strata are flagged; no thresholds are tuned on Luki.

In [ ]:
def score02(g):
    g=g[(g.qa=="ok")&g.mean_p.notna()].copy()
    if len(g)<2:return dict(n=len(g))
    y=g.label.to_numpy(dtype=int)
    p=g.mean_p.to_numpy(dtype=float)
    pred=(p>=CROWN_SCORE_THRESHOLD).astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1])
    tn,fp,fn,tp=cm.ravel()
    return dict(n=len(g),musanga=int(y.sum()),other=int((1-y).sum()),
        prevalence=float(y.mean()),mean_score=float(p.mean()),
        auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
        average_precision=float(average_precision_score(y,p)) if y.sum()>0 else np.nan,
        precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),
        specificity=float(tn/max(1,tn+fp)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)) if len(np.unique(y))==2 else np.nan,
        f1=float(f1_score(y,pred,zero_division=0)),
        brier=float(brier_score_loss(y,p)),
        tp=int(tp),tn=int(tn),fp=int(fp),fn=int(fn))
RESULTS=[]
for site in SITES:
    base=CROWN_PRED[CROWN_PRED.site==site]
    RESULTS.append(dict(site=site,stratum_type="all",stratum="all",
                        evaluation="independent_external" if site=="Luki" else "training_site_apparent",
                        **score02(base)))
    for typ in ["site_relative_light","illumination_stratum"]:
        for category,sub in base.groupby(typ):
            RESULTS.append(dict(site=site,stratum_type=typ,stratum=str(category),
                evaluation="independent_external" if site=="Luki" else "training_site_apparent",
                **score02(sub)))
METRICS=pd.DataFrame(RESULTS)
METRICS.to_csv(OUT/"02_frozen_crown_metrics_by_illumination.csv",index=False)
print("Whole-site metrics:")
display(METRICS[METRICS.stratum_type=="all"][["site","evaluation","n","musanga","other",
    "auc","average_precision","precision","recall","specificity","balanced_accuracy","f1",
    "fp","fn"]].round(3))
print("Luki relative illumination performance (interpret n and class prevalence):")
display(METRICS[(METRICS.site=="Luki")&
    (METRICS.stratum_type=="site_relative_light")].round(3))


## 05 — Error versus illumination and crown-level diagnostics

Test whether dark crowns have elevated false-positive scores, particularly **non-Musanga crowns in Luki**. Make density-free scatter and binned error summaries. No inference about cloud shadows is possible from a site-mean relationship alone.

In [ ]:
FIG,AX=plt.subplots(1,5,figsize=(21,4),constrained_layout=True)
for ax,site in zip(AX,SITES):
    df=CROWN_PRED[(CROWN_PRED.site==site)&(CROWN_PRED.qa=="ok")]
    for label,color,legend in [(0,"#356caa","Non-Musanga"),(1,"#d16e36","Musanga")]:
        z=df[df.label==label]
        ax.scatter(z.median_luminance,z.mean_p,c=color,s=10,alpha=.45,label=legend)
    ax.axhline(.5,c="black",linestyle="--",linewidth=1)
    ax.set(title=site,xlabel="Median crown RGB luminance",ylabel="Mean U-Net Musanga probability",
           xlim=(0,1),ylim=(0,1))
    if site=="Luki":ax.legend(fontsize=8)
FIG.savefig(OUT/"02_crown_probability_vs_illumination_by_site.png",dpi=170,bbox_inches="tight")
display(FIG);plt.close(FIG)

DIAGNOSTICS=[]
for (site,label),z in CROWN_PRED[CROWN_PRED.qa=="ok"].groupby(["site","label"]):
    rho=spearmanr(z.median_luminance,z.mean_p).statistic if len(z)>=10 and z.median_luminance.nunique()>1 else np.nan
    DIAGNOSTICS.append(dict(site=site,label=label,n=len(z),
        brightness_vs_musanga_score_spearman_rho=float(rho),
        median_probability=float(z.mean_p.median()),
        median_brightness=float(z.median_luminance.median())))
CORR=pd.DataFrame(DIAGNOSTICS)
CORR.to_csv(OUT/"02_crown_probability_brightness_correlations.csv",index=False)
display(CORR.round(3))
# Error class specificity: is darkness associated with elevated Musanga predictions on
# known non-Musanga crowns? This is stronger evidence than all-canopy prevalence alone.
LUKI_NEG=CROWN_PRED[(CROWN_PRED.site=="Luki")&
  (CROWN_PRED.label==0)&(CROWN_PRED.qa=="ok")]
LUKI_POS=CROWN_PRED[(CROWN_PRED.site=="Luki")&
  (CROWN_PRED.label==1)&(CROWN_PRED.qa=="ok")]
print("Luki FALSE-POSITIVE risk by within-site relative brightness (non-Musanga crowns):")
display(LUKI_NEG.groupby("site_relative_light").agg(
    crowns=("crown_id","count"),median_p=("mean_p","median"),
    mean_p=("mean_p","mean"),
    false_positive_rate=("mean_p",lambda x:float(np.mean(x>=.5)))).round(3))
print("Luki Musanga RECALL by within-site relative brightness:")
display(LUKI_POS.groupby("site_relative_light").agg(
    crowns=("crown_id","count"),median_p=("mean_p","median"),
    recall=("mean_p",lambda x:float(np.mean(x>=.5)))).round(3))


## 06 — Cross-site absolute RGB brightness: fixed common scale

**Primary quantitative comparison:** Use one universal, fixed 0–1 RGB luminance index across all five orthomosaics (the same Rec.709-weighted RGB formula already used in Notebook 01). Bin **every crown identically**, without estimating site-specific cutoffs. Summarize the occurrence of each brightness interval and model discrimination, false-positive and recall by interval and site. This permits literal comparison of recorded *image brightness*, but is **not an absolute irradiance measurement** because exposure, white balance and orthomosaic colour balancing differ across flights.

The within-site quartiles from Section 03 remain useful for cloud-shadow *anomalies* relative to a particular mosaic; they are secondary to these shared fixed bins. Class prevalence and very small bin sample sizes are explicitly shown. No model is trained, recalibrated or tuned here.

In [ ]:
# Exact common RGB luminance intervals, shared across all forests.
# Brightness index defined in V2/01: (.2126 R + .7152 G + .0722 B) / 255,
# using recorded uint8 image values. Units are IMAGE DIGITAL NUMBERS, NOT irradiance.
FIXED_EDGES02=np.linspace(0,1,11)
FIXED_LABELS02=[f"{FIXED_EDGES02[i]:.1f}–{FIXED_EDGES02[i+1]:.1f}"
                for i in range(len(FIXED_EDGES02)-1)]
CROWN_PRED["fixed_brightness_bin"]=pd.cut(
    CROWN_PRED.median_luminance,
    bins=FIXED_EDGES02,labels=FIXED_LABELS02,include_lowest=True,right=True)
CROWN_PRED.to_csv(OUT/"02_frozen_crown_predictions.csv",index=False)

# Exact same denominators and binning across forests. Report empty bins and both label classes.
FIXED_ROWS02=[]
for site in SITES:
    dat=CROWN_PRED[(CROWN_PRED.site==site)&(CROWN_PRED.qa=="ok")].copy()
    for band in FIXED_LABELS02:
        d=dat[dat.fixed_brightness_bin.astype(str)==band]
        pos=d[d.label==1];neg=d[d.label==0]
        FIXED_ROWS02.append(dict(
            site=site,brightness_bin=band,
            n_all=int(len(d)),n_musanga=int(len(pos)),n_other=int(len(neg)),
            musanga_prevalence=float(d.label.mean()) if len(d) else np.nan,
            mean_predicted_musanga_p=float(d.mean_p.mean()) if len(d) else np.nan,
            median_predicted_musanga_p=float(d.mean_p.median()) if len(d) else np.nan,
            false_positive_rate=float((neg.mean_p>=.5).mean()) if len(neg) else np.nan,
            musanga_recall=float((pos.mean_p>=.5).mean()) if len(pos) else np.nan,
            false_positive_n=int((neg.mean_p>=.5).sum()),
            false_negative_n=int((pos.mean_p<.5).sum()),
            model_evaluation="independent_external" if site=="Luki" else "training_site_apparent"))
FIXED_RESULTS02=pd.DataFrame(FIXED_ROWS02)
FIXED_RESULTS02.to_csv(OUT/"02_common_brightness_bins_metrics.csv",index=False)
print("Identical 0.1-wide brightness intervals used at all five sites.")
print("Crown counts per image-brightness band:")
display(FIXED_RESULTS02.pivot(index="brightness_bin",columns="site",values="n_all").fillna(0).astype(int))
print("False-positive rates within each brightness band (inspect n_other first):")
display(FIXED_RESULTS02.pivot(index="brightness_bin",columns="site",values="false_positive_rate").round(3))
print("Luki external evaluation by shared brightness bands:")
display(FIXED_RESULTS02[FIXED_RESULTS02.site=="Luki"].round(3))

# Common axes and bin breaks across forests (not percentile stretching).
fig,axs=plt.subplots(1,2,figsize=(15,5),constrained_layout=True)
for site in SITES:
    d=CROWN_PRED[(CROWN_PRED.site==site)&(CROWN_PRED.qa=="ok")]
    axs[0].hist(d.median_luminance,bins=FIXED_EDGES02,alpha=.55,
                histtype="step",linewidth=2,label=f"{site} (n={len(d)})")
    z=FIXED_RESULTS02[(FIXED_RESULTS02.site==site)&(FIXED_RESULTS02.n_other>=5)]
    xm=np.array([float(s.split("–")[0])+.05 for s in z.brightness_bin])
    axs[1].plot(xm,z.false_positive_rate,marker="o",label=site)
axs[0].set(xlim=(0,1),xlabel="Recorded RGB luminance index (shared 0–1 scale)",
           ylabel="Number of annotated crowns",
           title="Cross-site crown brightness distribution")
axs[1].set(xlim=(0,1),ylim=(0,1),xlabel="Recorded RGB luminance index (shared 0–1 scale)",
           ylabel="Non-Musanga false-positive rate",
           title="False positives by absolute image-brightness bin (n ≥5 negatives)")
for ax in axs:ax.legend(fontsize=8)
fig.savefig(OUT/"02_cross_site_absolute_brightness_vs_false_positives.png",
            dpi=170,bbox_inches="tight")
display(fig);plt.close(fig)

# A site-independent numerical comparison of the RGB brightness values.
# Note that camera/orthomosaic processing may prevent interpretation as illumination exposure.
CROSS_SUM02=CROWN_PRED.groupby("site").agg(
    crowns=("crown_id","count"),
    crowns_with_prediction=("mean_p","count"),
    median_RGB_index=("median_luminance","median"),
    p10_RGB_index=("median_luminance",lambda v:v.quantile(.10)),
    p90_RGB_index=("median_luminance",lambda v:v.quantile(.90)),
    dark_under_0p4_pct=("median_luminance",lambda v:100*float((v<.4).mean())),
    bright_over_0p6_pct=("median_luminance",lambda v:100*float((v>=.6).mean())),
    mean_predicted_score=("mean_p","mean")).reset_index()
CROSS_SUM02.to_csv(OUT/"02_cross_site_absolute_brightness_summary.csv",index=False)
display(CROSS_SUM02.round(3))
print("Warning: cross-site image brightness is not radiometrically calibrated irradiance.")
print("Original sources and U-Net maps unchanged. New outputs:",OUT)


## 07 — Provenance, limitations and handoff to robust training

Freeze results, baseline threshold and illumination cutoffs. The next experiment should compare photometric augmentation against the same independent Luki reference without adjusting settings to maximize Luki performance.

In [ ]:
manifest=dict(
    stage="V2/02 Frozen U-Net crown-level and illumination audit",
    source_maps=str(MAPS),reference_source=str(MERGED),
    crown_probability_score="mean valid frozen U-Net P(Musanga) within each labelled crown",
    pixel_cover_diagnostic="fraction of valid crown pixels with U-Net p >=0.5",
    mean_probability_threshold=CROWN_SCORE_THRESHOLD,
    min_valid_pixels=MIN_CROWN_VALID_PIXELS,min_valid_fraction=.90,
    retained_luki_crowns=int((CROWN_PRED.site=="Luki").sum()),
    luki_positive=int(((CROWN_PRED.site=="Luki")&(CROWN_PRED.label==1)).sum()),
    luki_negative=int(((CROWN_PRED.site=="Luki")&(CROWN_PRED.label==0)).sum()),
    luki_reference_version=LUKI_VERSION,
    external_evaluation="Luki only; four original training forests were used to train frozen production U-Net",
    site_relative_strata="quartiles of crown RGB luminance within each forest",
    cross_site_brightness="fixed 0.1-width bins of shared native RGB digital-number luminance, not radiometric irradiance",
    threshold_selection="not tuned on Luki",
    native_uav_crs="unchanged",outputs=str(OUT))
(OUT/"02_frozen_baseline_manifest.json").write_text(json.dumps(manifest,indent=2))
print("V2 NOTEBOOK 02 COMPLETE:",OUT)
print("NEXT: V2 Notebook 03 — train illumination-robust U-Net on the four original forests.")
print("Do not interpret four training-site apparent metrics as LOSO validation.")
